# GeoPulse EDA

Run `make phase1` first so the artifacts below exist.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import polars as pl

from src.utils.config import load_config, resolve_path

cfg = load_config()
interim = resolve_path(cfg, "paths.interim")
external = resolve_path(cfg, "paths.external")
spatial = resolve_path(cfg, "paths.spatial")
trips = pl.scan_parquet(interim / "trips_clean.parquet")
trips.head().collect()

## Rides per day, and the weekly/seasonal shape

In [ ]:
tz = cfg.dotted("time.timezone")
daily = (
    trips.with_columns(pl.col("started_at").dt.convert_time_zone(tz).dt.date().alias("day"))
    .group_by("day").agg(pl.len().alias("rides")).sort("day").collect()
)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily["day"], daily["rides"], lw=0.8)
ax.set_title("Citi Bike rides per day, 2023-2024"); ax.set_ylabel("rides")
plt.tight_layout()
daily.describe()

## Demand by hour of day, weekday vs weekend

In [ ]:
hourly = (
    trips.with_columns(pl.col("started_at").dt.convert_time_zone(tz).alias("local"))
    .with_columns([
        pl.col("local").dt.hour().alias("hour"),
        (pl.col("local").dt.weekday() >= 6).alias("is_weekend"),
    ])
    .group_by(["hour", "is_weekend"]).agg(pl.len().alias("rides")).sort("hour").collect()
)
fig, ax = plt.subplots(figsize=(9, 4))
for weekend, label in [(False, "weekday"), (True, "weekend")]:
    part = hourly.filter(pl.col("is_weekend") == weekend)
    ax.plot(part["hour"], part["rides"], marker="o", label=label)
ax.set_xlabel("hour (America/New_York)"); ax.set_ylabel("rides"); ax.legend()
plt.tight_layout()

## Ride duration and trip-distance sanity

In [ ]:
durations = trips.select("ride_duration").collect()["ride_duration"]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(durations.filter(durations < 3600).to_numpy(), bins=120)
ax.set_xlabel("ride duration (s, < 1h)"); ax.set_ylabel("rides")
plt.tight_layout()
durations.describe()

## Weather join sanity: demand vs rain

In [ ]:
weather = pl.read_parquet(external / "weather_hourly.parquet")
hourly_rides = (
    trips.with_columns(pl.col("started_at").dt.truncate("1h").alias("weather_timestamp"))
    .group_by("weather_timestamp").agg(pl.len().alias("rides")).collect()
)
joined = hourly_rides.join(weather, on="weather_timestamp", how="inner").with_columns(
    (pl.col("precipitation") > cfg.dotted("weather.rain_mm_threshold")).alias("is_raining")
)
joined.group_by("is_raining").agg([pl.len().alias("hours"), pl.col("rides").mean().alias("mean_rides")])

## Station registry: where the network is, and what got flagged

In [ ]:
registry = pl.read_parquet(spatial / "station_registry.parquet")
fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(registry["lng"], registry["lat"], s=registry["appearances"] / registry["appearances"].max() * 40, alpha=0.5)
ax.set_title(f"{registry.height:,} stations, sized by trip volume"); ax.set_aspect(1.32)
plt.tight_layout()
registry.filter(pl.col("coord_anomaly")).select(
    ["station_id", "station_name", "coord_spread_km_robust", "appearances"]
).head(20)